# Valuation of Performance Share Options (PSOs) under IFRS 2: A Theoretical Note

## 1. Conceptual Framework
Performance Share Options (PSOs) are equity-based awards whose ultimate settlement value depends on both the underlying share price and a set of vesting conditions. Under IFRS 2, the grant-date fair value of a PSO is determined by incorporating market-based option characteristics in the initial valuation, while treating service and non-market performance conditions through probabilistic vesting adjustments.

A PSO can be mathematically represented as a contract with the following payoff structure. For an equity-settled award, the payoff $H$ is:
$$H = \max(S_T - K, 0) \cdot g(S_{\cdot}, M_{\cdot}) \cdot 1_{\{\text{Vesting}\}}$$

Alternatively, if the award is expressed as a dynamically adjusted number of delivered options, the payoff becomes:
$$H = N(S_{\cdot}, M_{\cdot}) \cdot \max(S_T - K, 0)$$

[cite_start]To accommodate potential early exercise features (American-style options) and discrete dividend payments, the valuation typically employs a Cox-Ross-Rubinstein (CRR) binomial lattice. Over a time step $\Delta t = T / N$, the underlying price evolution is governed by the up-factor $u$, the down-factor $d$, and the risk-neutral probability $p$:
$$u = \exp(\sigma\sqrt{\Delta t}), \quad d = \exp(-\sigma\sqrt{\Delta t}), \quad p = \frac{\exp((r-q)\Delta t) - d}{u - d}$$

The option is valued via backward induction. At any node $(i, j)$, the continuation value is:
$$V_{i,j} = \exp(-r\Delta t) \left[ p \, V_{i+1,j+1} + (1-p) \, V_{i+1,j} \right]$$

The terminal node values at maturity ($N$) incorporate the intrinsic option value alongside the path-dependent and non-market multipliers:
$$V_{N,j} = \max(S_{N,j} - K, 0) \cdot g(S_{\cdot}, M_{\cdot}) \cdot 1_{\{\text{Vesting}\}}$$

This lattice structure clarifies the methodological bifurcation inherent in IFRS 2: the market-based option component is priced under the risk-neutral measure $Q$ via the CRR tree, whereas the multiplicative performance factor $g(\cdot)$ and the indicator $1_{\{\text{Vesting}\}}$ capture the real-world probabilistic impact of service and non-market vesting conditions. 

Consequently, the aggregate fair value is recognized over the requisite service period. Updates to service forfeiture estimates or non-market condition attainability prospectively adjust the recognized expense, while market conditions remain crystallized at the grant-date valuation.

## 2. Implementation Considerations
Key implementation principles include:
* Calibrating the underlying risk-neutral dynamics to market-implied volatility and dividend yields.
* Modeling non-market metrics under realistic distributions or empirical scenario analyses.
* Incorporating correlation structures between the underlying equity and non-market drivers, where material dependencies exist.
* Verifying the numerical stability of the lattice with sensitivity tests on volatility, interest rates, correlations, and vesting hazard rates.

In [ ]:
import numpy as np
from scipy.stats import norm

def calculate_pso_binomial_tree(S0, K, T, T_vesting, r, q, sigma, N_steps):
    """
    Valuta una Performance Share Option (PSO) con finestra di esercizio ritardata
    utilizzando il modello binomiale di Cox-Ross-Rubinstein.
    """
    # 1. Parametri CRR
    dt = T / N_steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp((r - q) * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    
    # 2. Inizializzazione dei prezzi del sottostante a scadenza (Nodo N)
    # Vettore dei prezzi possibili al tempo T
    S_T = S0 * (d ** np.arange(N_steps, -1, -1)) * (u ** np.arange(0, N_steps + 1, 1))
    
    # 3. Payoff Intrinseco a scadenza
    V = np.maximum(S_T - K, 0)
    
    # 4. Induzione a Ritroso (Backward Induction)
    for i in range(N_steps - 1, -1, -1):
        # Vettore dei prezzi del sottostante al nodo temporale i
        S_i = S0 * (d ** np.arange(i, -1, -1)) * (u ** np.arange(0, i + 1, 1))
        
        # Valore di continuazione (attualizzazione dei due nodi futuri)
        V_cont = discount * (p * V[1:i+2] + (1 - p) * V[0:i+1])
        
        # Tempo corrente in anni dall'inizio
        t_current = i * dt
        
        # Condizione logica per la finestra di esercizio
        if t_current >= T_vesting:
            # Finestra aperta (Stile Americano): il manager può esercitare se conviene
            V = np.maximum(V_cont, S_i - K)
        else:
            # Finestra chiusa (Stile Europeo): il manager deve mantenere l'opzione
            V = V_cont
            
    return V[0] # Ritorna il Fair Value alla radice dell'albero (Grant Date)

def calculate_finnerty_dlom(sigma, T_lockup, q):
    """
    Calcola il moltiplicatore DLOM secondo il modello di Finnerty (2012).
    Rappresenta il valore trattenuto (1 - Sconto).
    """
    # Varianza cumulativa sul periodo di lock-up
    v = sigma * np.sqrt(T_lockup)
    
    # Parametri di approssimazione per la Put asiatica
    d1 = (v / 2)
    d2 = -(v / 2)
    
    # Costo percentuale dell'illiquidità
    # (Formula standardizzata per lockup senza dividendi eccessivi)
    discount = np.exp(-q * T_lockup) * norm.cdf(d1) - norm.cdf(d2)
    
    # Ritorna il moltiplicatore da applicare al Fair Value
    return max(1.0 - discount, 0.0)

# --- INPUT DATI DAL VALUATION REPORT ---
S0_pso = 97.70          # Spot price Exor al 1-Lug-24
K = 108.09              # Strike Price con hurdle rate
sigma = 0.1599          # Volatilità 15.99%
q = 0.0045              # Dividend Yield PSO 0.45%
r = 0.03                # Tasso Risk-Free (Assunto)

# Finestre Temporali (in anni dal Grant Date)
# Dal 01-Jul-24 al 31-Dec-30 = 6.5 anni circa
T_maturity = 6.5 
# Dal 01-Jul-24 al 01-Jan-27 = 2.5 anni circa
T_vesting_cliff = 2.5   
# Lock-up post esercizio per CEO
T_lockup_ceo = 2.0      

# Risoluzione dell'albero (1000 step garantiscono precisione accademica)
N_steps = 1000

# --- ESECUZIONE ---
print("Calcolo Fair Value PSO (Binomial Tree CRR)...")

# 1. Calcolo del Fair Value Base (Prima del DLOM)
fv_pso_base = calculate_pso_binomial_tree(
    S0=S0_pso, K=K, T=T_maturity, T_vesting=T_vesting_cliff, 
    r=r, q=q, sigma=sigma, N_steps=N_steps
)

# 2. Applicazione del DLOM per il CEO
dlom_multiplier = calculate_finnerty_dlom(sigma=sigma, T_lockup=T_lockup_ceo, q=q)
fv_pso_ceo = fv_pso_base * dlom_multiplier

print("=" * 60)
print("RISULTATI VALUTAZIONE PSO IFRS 2")
print("=" * 60)
print(f"Fair Value PSO (Other Participants): € {fv_pso_base:.4f}")
print(f"DLOM Discount (Finnerty 2Y Lockup):  {(1 - dlom_multiplier):.2%}")
print(f"Fair Value PSO (CEO Profile):        € {fv_pso_ceo:.4f}")
print("=" * 60)